## 📖 Libro: §2.1–2.2 del Capítulo 2 — KL y $f$-divergencias (KL asimetria + JSD acotada)

**Enunciado (verbatim del libro):** *"La divergencia KL NO es simétrica, D(p\|q) ≠ D(q\|p), y NO satisface la desigualdad triangular... la asimetría es una propiedad fundamental, no un defecto."*

**Mini-reto:**
1. Verificar `D_KL(p\|q) ≠ D_KL(q\|p)` y graficar la asimetría sobre pares Bernoulli.
2. Verificar que JSD `= ½ D_KL(p\|m) + ½ D_KL(q\|m)` satisface `0 ≤ JSD ≤ ln(2)` (acotada, simétrica).
3. Visualizar la divergencia KL como función de `p` para `q=0.5` fijo en Bernoulli.
4. Cross-check: la KL cumple `KL(p\|q) = ψ(η_q) − ψ(η_p) − ⟨η_q−η_p, ∇ψ(η_p)⟩` (divergencia de Bregman).

**@ Pregunta a tu LLM:** «¿Cuál es la intuición geométrica de la asimetría KL? Si KL mide "el costo de describir p como q", ¿por qué ese costo no es recíproco?»

In [ ]:
# =====================================================================
# Celda 1 — imports + seed determinístico
# =====================================================================
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt

from utils import setup_seed

SEED = setup_seed("cap2_kl_jensen_shannon")
rng = np.random.default_rng(SEED)
print(f"Deterministic seed: {SEED}")

In [ ]:
# =====================================================================
# Celda 2 — Implementación vectorizada de KL y JSD (Bernoulli + Categorical)
# =====================================================================
def kl_bernoulli(p: float, q: float) -> float:
    """D_KL(Bern(p) || Bern(q)); robusto en los bordes."""
    p = np.clip(p, 1e-15, 1-1e-15)
    q = np.clip(q, 1e-15, 1-1e-15)
    return float(p * np.log(p / q) + (1 - p) * np.log((1 - p) / (1 - q)))

def jsd_bernoulli(p: float, q: float, base: float = 2.0) -> float:
    """Jensen-Shannon en bits si base=2, en nats si base=e."""
    m = 0.5 * (p + q)
    val = 0.5 * kl_bernoulli(p, m) + 0.5 * kl_bernoulli(q, m)
    if base == 2.0:
        return val / np.log(2)
    return val

def kl_categorical(p: np.ndarray, q: np.ndarray) -> float:
    p = np.clip(p, 1e-15, None); p = p / p.sum()
    q = np.clip(q, 1e-15, None); q = q / q.sum()
    return float(np.sum(p * np.log(p / q)))

def jsd_categorical(p: np.ndarray, q: np.ndarray) -> float:
    m = 0.5 * (p + q)
    return 0.5 * kl_categorical(p, m) + 0.5 * kl_categorical(q, m)

print("Verificación de simetría Bernoulli(0.7) vs Bernoulli(0.3):")
print(f"  D_KL(0.7 || 0.3) = {kl_bernoulli(0.7, 0.3):.4f}")
print(f"  D_KL(0.3 || 0.7) = {kl_bernoulli(0.3, 0.7):.4f}")
print(f"  -> Asimetría: D_KL(0.7||0.3) - D_KL(0.3||0.7) = "
      f"{kl_bernoulli(0.7, 0.3) - kl_bernoulli(0.3, 0.7):.4f}")
print(f"  -> JSD bits: {jsd_bernoulli(0.7, 0.3, base=2.0):.4f} (debe ∈ [0, 1])")

In [ ]:
# =====================================================================
# Celda 3 — Asimetría KL sobre una nube aleatoria de pares Bernoulli
# =====================================================================
# Para N pares (p, q) aleatorios, grafica D(p||q) contra D(q||p).
# Diagonal y = x marca simetría; las asimetrías muestran arriba/abajo.
N = 1000
ps = rng.uniform(0.05, 0.95, N)
qs = rng.uniform(0.05, 0.95, N)
kl_pq = np.array([kl_bernoulli(p, q) for p, q in zip(ps, qs)])
kl_qp = np.array([kl_bernoulli(q, p) for p, q in zip(ps, qs)])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(kl_pq, kl_qp, s=10, alpha=0.4, color="#0F766E")
xx = np.linspace(0, max(kl_pq.max(), kl_qp.max()), 100)
axes[0].plot(xx, xx, "black", linestyle="--", linewidth=1, label="identidad (simetría)")
axes[0].set_xlabel("D_KL(p || q)")
axes[0].set_ylabel("D_KL(q || p)")
axes[0].set_title("¿D_KL(p||q) = D_KL(q||p)? (NO: únicamente en p = q)")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].hist(kl_pq - kl_qp, bins=50, color="#8a2be2", alpha=0.7,
             label=r"$D_{KL}(p||q) - D_{KL}(q||p)$")
axes[1].axvline(0, color="black", linestyle="--", linewidth=1)
axes[1].set_xlabel("asimetría [nats]")
axes[1].set_ylabel("frecuencia")
axes[1].set_title("Distribución de la asimetría KL sobre 1000 pares aleatorios")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# Celda 4 — Visualización de KL(p || q) vs KL(q || p) para q=0.5 fijo
# =====================================================================
# La asimetría es clara: KL(p||0.5) tiene su mínimo en p=0.5 pero
# ambas curvas son convexas en p pero asimétricas entre sí.
ps = np.linspace(0.005, 0.995, 500)
kl_p_05 = np.array([kl_bernoulli(p, 0.5) for p in ps])
kl_05_p = np.array([kl_bernoulli(0.5, p) for p in ps])
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(ps, kl_p_05, color="#0F766E", linewidth=2.5,
        label=r"$D_{KL}(\mathrm{Bern}(p) \| \mathrm{Bern}(0.5))$",
        marker="o", markevery=80, markersize=6)
ax.plot(ps, kl_05_p, color="#ff5fd2", linewidth=2.5, linestyle="--",
        label=r"$D_{KL}(\mathrm{Bern}(0.5) \| \mathrm{Bern}(p))$",
        marker="s", markevery=80, markersize=6)
ax.axvline(0.5, color="black", linestyle=":", linewidth=1)
ax.fill_between(ps, kl_p_05, kl_05_p, alpha=0.18, color="gray",
                label="asimetría")
ax.set_xlabel("p")
ax.set_ylabel("KL [nats]")
ax.set_title("Asimetría KL: D_KL(p||q) vs D_KL(q||p) para q=0.5 fijo")
ax.legend(loc="upper center")
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.2)
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# Celda 5 — JSD acotada: para Categorical k-sym °1.verifica 0 ≤ JSD ≤ ln 2
# =====================================================================
# JSD bits está en [0, 1] y JSD nats en [0, ln 2]. Verifica numéricamente.
k = 3  # Categorical de 3 categorías
n_trials = 5000
max_jsd_nats = 0
min_jsd_nats = np.inf
rnd_p = rng.dirichlet(np.ones(k), size=n_trials)
rnd_q = rng.dirichlet(np.ones(k), size=n_trials)
for p, q in zip(rnd_p, rnd_q):
    j = jsd_categorical(p, q)
    max_jsd_nats = max(max_jsd_nats, j)
    min_jsd_nats = min(min_jsd_nats, j)
print(f"JSD nats: mín = {min_jsd_nats:.6f}, máx = {max_jsd_nats:.6f}")
print(f"JSD nats teórico [0, ln 2]  = [0, {np.log(2):.6f}]")
print(f"¿Cumple la cota?: "
      f"min >= 0: {min_jsd_nats >= -1e-9}, "
      f"max <= ln 2: {max_jsd_nats <= np.log(2) + 1e-9}")

# Igual chequeo en bits
max_jsd_bits = 0; min_jsd_bits = np.inf
for p, q in zip(rnd_p, rnd_q):
    j = jsd_categorical(p, q) / np.log(2)
    max_jsd_bits = max(max_jsd_bits, j)
    min_jsd_bits = min(min_jsd_bits, j)
print(f"\nJSD bits: mín = {min_jsd_bits:.6f}, máx = {max_jsd_bits:.6f}")
print(f"JSD bits teórico [0, 1]")
print(f"¿Cumple la cota?: min ≥ 0 y max ≤ 1 con tolerancia 1e-9: "
      f"{min_jsd_bits >= -1e-9 and max_jsd_bits <= 1 + 1e-9}")

In [ ]:
# =====================================================================
# Celda 6 — Verificación cross-check: KL = Bregman en coordenadas naturales
# =====================================================================
# Para Bernoulli: A(η) = log(1 + e^η), ∇A(η) = p. La KL entre dos puntos
# cumple D_KL(p_1||p_2) = A(η_2) - A(η_1) - ⟨η_2 - η_1, p_1⟩.
print("Cross-check: KL = Bregman(A) en coordenadas η")
print(f"{'p1':>5s} {'p2':>5s} {'η1':>8s} {'η2':>8s} "
      f"{'KL(p1||p2)':>14s} {'Bregman':>14s} {'diff':>10s}")
print("-" * 70)
for p1, p2 in [(0.7, 0.3), (0.9, 0.1), (0.5, 0.5), (0.2, 0.8), (0.99, 0.01)]:
    eta1 = np.log(p1 / (1 - p1))
    eta2 = np.log(p2 / (1 - p2))
    A_eta1 = np.log1p(np.exp(eta1))
    A_eta2 = np.log1p(np.exp(eta2))
    grad_A_eta1 = p1  # ∇A(η) = p por Bernoulli
    bregman = A_eta2 - A_eta1 - (eta2 - eta1) * grad_A_eta1
    kl = kl_bernoulli(p1, p2)
    print(f"{p1:>5.2f} {p2:>5.2f} {eta1:>8.4f} {eta2:>8.4f} "
          f"{kl:>14.6f} {bregman:>14.6f} {abs(kl-bregman):>10.2e}")

# Esperado: diff ~ 1e-10 en cada caso (error de doble precisión).

## ✅ `@ Verifica con:`

Las verificaciones que se cumplen:

1. **Asimetría visible**: el panel izquierdo del scatter muestra (KL_pq, KL_qp) distribuido ASIM\u00c9TRICAMENTE alrededor de la diagonal; el panel derecho muestra la distribución de (KL_pq − KL_qp) centrada pero extendida en ambos lados (no exactamente en 0).
2. **Asimetría visual función de p**: la gráfica con p=variable y q=0.5 muestra las curvas NO coincidentes; el área entre ambas es la asimetría.
3. **JSD acotada**: para 5000 pares aleatorios de Categorical(3), JSD_nats cumple `[0, ln 2 ≈ 0.693]` y JSD_bits cumple `[0, 1]` con tolerancia 1e-9.
4. **Cross-check Bregman**: la columna `diff` debe ser ≤ 1e-10 para todos los pares (p1, p2), confirmando `D_KL = A(η2) - A(η1) - ⟨η2−η1, ∇A(η1)⟩`.

Conexión con el libro:
- §2.1 (KL): la asimetría `D(p||q) ≠ D(q||p)` es un teorema explícito. La verificación numérica aquí confirma empíricamente.
- §2.2 (f-divergencias + JSD): JSD es la única “común” simétrica; acotada [0, ln 2] permite interpretaciones como probabilidad de detección.
- §2.3 (Bregman): la celda 6 confirma numéricamente que KL ≡ Bregman(A) en coordenadas naturales para la familia exponencial Bernoulli.

Discusión para el LLM mentor:
- ¿Por qué la KL diverge a infinito en los bordes (p→0 con q=0.5)? Porque p log(p/q) → ∞ cuando p→0.
- ¿Hay alguna divergencia *simétrica* y *acotada* además de JSD? Sí: Hellinger (acotada a (0, √2)), variaci\u00f3n total (acotada a (0, 2)).
- ¿Por qué `Bregman(A)` no es sim\u00e9trico en general sino por el logit si? Porque la transformada `p → η = logit(p)` no preserva la noción de distancia euclidiana (es una curvatura intr\u00ednseca del simplex).